# SAC training

Pre-fix outputs are preserved in `../archive/notebooks/` under this section name. These updated cells have no historical execution output. Run with the project Python environment. Training is opt-in; old models are never loaded automatically.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rl_project").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Image, Video, display
from Continuous_Diff_Drive.experiments import create_agent, DEFAULTS, ENVIRONMENT, EPISODES


## Configuration
The environment samples a fixed mixture of obstacle families. The mode name `curriculum` does not imply increasing difficulty. Normalization is fixed physical scaling; cutoffs retain a bootstrap and carry no extra reward penalty.

In [ ]:
ALGORITHM = "sac"
RUN_TRAINING = False
SEED = 0
NUM_EPISODES = EPISODES[ALGORITHM]
MAX_ENV_STEPS = None
OUTPUT_ROOT = ROOT / "artifacts"
RECORD_EVERY = None  # Set to 500 for optional separate greedy demonstrations.
WARM_START_CHECKPOINT = None  # Explicit new-format checkpoint; never an exact resume.
ENV_KWARGS = {**ENVIRONMENT, "max_step": 1200}
AGENT_KWARGS = DEFAULTS[ALGORITHM].copy()


## Train
The saved model and incremental CSV precede optional plotting. A warm start keeps learned network weights but resets replay, optimizers, exploration, temperature and counters. SAC requires a new normalized-input run; the old SAC checkpoint is incompatible.

In [ ]:
agent = None
result = None
if RUN_TRAINING:
    agent = create_agent(ALGORITHM, seed=SEED, env_kwargs=ENV_KWARGS, agent_kwargs=AGENT_KWARGS)
    if WARM_START_CHECKPOINT is not None:
        agent.load_checkpoint(WARM_START_CHECKPOINT, mode="warm_start")
    result = agent.train(NUM_EPISODES, max_env_steps=MAX_ENV_STEPS,
                         output_root=OUTPUT_ROOT, record_every=RECORD_EVERY)
    print(result.artifacts)
else:
    print("Training not run. Set RUN_TRAINING=True to start a fresh run; no checkpoint was loaded.")


## Inspect the new run
Bands show rolling mean ± two rolling standard deviations within this run. The outcome plot separates binary success from the rolling success rate; losses remain aligned with real episode numbers. Videos are at most 320 pixels wide and 300 frames long.

In [ ]:
if result is not None:
    for path in sorted((result.run_dir / "plots").glob("*.png")):
        display(Image(filename=str(path)))
    for path in sorted((result.run_dir / "videos/training").glob("*.mp4")):
        display(Video(filename=str(path), embed=True))


For evaluation, use the corresponding evaluation notebook and select the new checkpoint explicitly, or select the latest completed new run. For five seeds with matched environment-step budgets, use the optional comparison runner described in the section README. No full experiments were rerun as part of the implementation fixes.